# Visualize Training Results

Use this notebook to inspect a training run folder from `runs/`. It can load `metrics.csv`, `config.json`, and available checkpoints, then plot loss and Dice curves.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNS_ROOT = PROJECT_ROOT / "runs"
RUNS_ROOT.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Runs root: {RUNS_ROOT}")

## Select Run Folder

In [ ]:
def find_run_folders(runs_root):
    folders = []
    for folder in sorted(runs_root.iterdir()):
        if folder.is_dir() and (folder / "metrics.csv").exists():
            folders.append(folder)
    return folders


run_folders = find_run_folders(RUNS_ROOT)

if not run_folders:
    selected_run_dir = None
    print("No run folders with metrics.csv found yet.")
else:
    selected_run_dir = run_folders[-1]

    if widgets is None:
        print("ipywidgets is not available. Set selected_run_dir manually, for example:")
        print("selected_run_dir = RUNS_ROOT / 'fcn8_5epoch_gpu_bs16_lr1e-3'")
        print(f"Defaulting to latest folder: {selected_run_dir}")
    else:
        options = [(folder.relative_to(PROJECT_ROOT).as_posix(), folder) for folder in run_folders]
        dropdown = widgets.Dropdown(
            options=options,
            value=selected_run_dir,
            description="Run folder:",
            layout=widgets.Layout(width="700px"),
        )
        output = widgets.Output()

        def update_selected_run(change=None):
            global selected_run_dir
            selected_run_dir = dropdown.value
            with output:
                output.clear_output()
                print(f"Selected: {selected_run_dir.relative_to(PROJECT_ROOT)}")

        dropdown.observe(update_selected_run, names="value")
        display(dropdown, output)
        update_selected_run()

## Load Metrics And Config

In [ ]:
if selected_run_dir is None:
    raise RuntimeError("Select or create a run folder first.")

metrics_path = selected_run_dir / "metrics.csv"
config_path = selected_run_dir / "config.json"

metrics = pd.read_csv(metrics_path)
config = json.loads(config_path.read_text()) if config_path.exists() else {}

print(f"Loaded: {metrics_path.relative_to(PROJECT_ROOT)}")
print(f"Epochs logged: {len(metrics)}")
print(f"Device: {config.get('device', 'unknown')}")
print(f"Train files: {config.get('num_train_files', 'unknown')}")
print(f"Validation files: {config.get('num_val_files', 'unknown')}")

metrics.tail()

## Loss And Dice Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(metrics["epoch"], metrics["train_loss"], marker="o", label="train")
axes[0].plot(metrics["epoch"], metrics["val_loss"], marker="o", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross entropy")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(metrics["epoch"], metrics["train_mean_foreground_dice"], marker="o", label="train")
axes[1].plot(metrics["epoch"], metrics["val_mean_foreground_dice"], marker="o", label="validation")
axes[1].set_title("Mean Foreground Dice")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

## Per-Class Dice

In [ ]:
val_dice_columns = [column for column in metrics.columns if column.startswith("val_dice_class_")]
train_dice_columns = [column for column in metrics.columns if column.startswith("train_dice_class_")]

if not val_dice_columns:
    print("No per-class Dice columns found.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for column in train_dice_columns:
        class_id = column.rsplit("_", 1)[-1]
        axes[0].plot(metrics["epoch"], metrics[column], marker="o", label=f"class {class_id}")
    axes[0].set_title("Train Dice Per Class")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Dice")
    axes[0].set_ylim(0, 1)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    for column in val_dice_columns:
        class_id = column.rsplit("_", 1)[-1]
        axes[1].plot(metrics["epoch"], metrics[column], marker="o", label=f"class {class_id}")
    axes[1].set_title("Validation Dice Per Class")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()

## Best Epoch Summary

In [ ]:
best_index = metrics["val_mean_foreground_dice"].idxmax()
best_row = metrics.loc[best_index]

summary_columns = [
    "epoch",
    "train_loss",
    "train_mean_foreground_dice",
    "val_loss",
    "val_mean_foreground_dice",
    "epoch_seconds",
    "elapsed_time",
]
summary_columns = [column for column in summary_columns if column in metrics.columns]

print("Best validation Dice epoch:")
display(best_row[summary_columns].to_frame(name="value"))

## Checkpoints

In [ ]:
checkpoint_rows = []
for checkpoint in sorted(selected_run_dir.glob("*.pt")):
    checkpoint_rows.append({
        "checkpoint": checkpoint.name,
        "size_mb": checkpoint.stat().st_size / (1024 * 1024),
        "path": checkpoint.relative_to(PROJECT_ROOT).as_posix(),
    })

if checkpoint_rows:
    display(pd.DataFrame(checkpoint_rows))
else:
    print("No checkpoint files found in this run folder.")

## Run Configuration

In [ ]:
if config:
    print(json.dumps(config, indent=2))
else:
    print("No config.json found in this run folder.")